# Geographic storytelling and honest GIS handoffs

How can a story preserve evidence while guiding attention? Build a three-panel earthquake narrative and export a small, inspectable GIS handoff. The exchange format travels across runtimes; native GIS packages do not thereby run in Pyodide.

## Objectives

- Keep a story's selected counts and coordinate assumptions visible
- Export GeoJSON, CSV and a self-contained HTML evidence summary
- Distinguish a browser lesson from external GIS and native GeoAI continuations

## Start in a fresh kernel

Run every code cell from the top. The Book shows publication-time outputs;
JupyterLite uses the Python (Pyodide) kernel and downloads its runtime and
packages initially. All observations and metadata below come from packaged
snapshots: no live API, token, paid service or GPU is required. If a file is
missing or modified, validation fails explicitly; restore the supplied files.
Download notebook edits and exports to keep them outside browser storage.
The Python code is visible so that transformations can be inspected.

In [ ]:
import sys
from pathlib import Path

if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib"])

# Book starts here; desktop/Lite may start at either the notebook or content root.
candidates = [Path.cwd(), Path.cwd() / "repository_examples",
              Path.cwd() / "content" / "repository_examples"]
lesson_dir = next((p for p in candidates if (p / "repo_examples.py").is_file()), None)
if lesson_dir is None:
    raise FileNotFoundError("Open this lesson from content/repository_examples on desktop, or repository_examples in Lite, with repo_examples.py and data/ alongside it.")
sys.path.insert(0, str(lesson_dir))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import repo_examples as rx

if sys.platform != "emscripten":
    get_ipython().run_line_magic("matplotlib", "inline")

plt.rcParams.update({"figure.figsize": (9, 4.8), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
display(rx.validate_all())

## Evidence and attribution

The cards describe the evidence type before any visualization. The manifest retains exact URLs/queries, retrieval UTC, SHA-256 checksums, field order, row counts and transformation recipes. Dataset terms remain separate from the undecided project license.

In [ ]:
display(rx.source_card("usgs"))
display(rx.source_card("atlas"))

In [ ]:
import base64
import csv
import io
import json
from html import escape

events = rx.read_csv("usgs/events.csv")
events["time_utc"] = pd.to_datetime(events.time_utc, utc=True)
atlas = rx.read_csv("repository_atlas.csv")
selected = events.loc[events.magnitude >= 5].copy()
display(pd.DataFrame({"stage": ["Packaged selection", "Story emphasis", "Depth discussion"],
                      "rule": ["January 2024, M≥4.5, fixed Japan bbox", "Reported magnitude ≥5", "Same ≥5 records; depth is positive down"],
                      "events": [len(events), len(selected), len(selected)]}))

## View 1: a narrative with explicit filters

The first panel locates all selected epicenters. The second asks when higher-magnitude events occurred. The third retains that second selection and reveals depth. Each panel answers one question and declares its denominator. These are editorial stages, not a model of a geological process or a forecast.

**Method connection:** the catalog's Story template (§58) and Flow chart (§26) organize the narrative and exchange sequence, while its Table (§3) and Scatterplot (§9) retain inspectable evidence. Geographic story panels and small multiples are additional compound constructions. The source repositories offer patterns for story sequencing and exchange; their unlicensed authored projects, invented risk fields and unverified GIS integrations are not imported.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].scatter(events.longitude, events.latitude, s=18, alpha=.6, color="#2b7792")
rx.finish(axes[0], f"1. Where? n={len(events)}", "Longitude (° E)", "Latitude (° N)")
axes[1].scatter(selected.time_utc, selected.magnitude, color="#b67638", s=25)
rx.finish(axes[1], f"2. When? M≥5, n={len(selected)}", "January 2024 UTC", "Reported magnitude")
axes[1].tick_params(axis="x", labelrotation=45)
axes[2].scatter(selected.longitude, selected.depth_km, color="#2b7792", s=25)
axes[2].invert_yaxis()
rx.finish(axes[2], f"3. How deep? n={len(selected)}", "Longitude (° E)", "Catalog depth (km, positive down)")
fig.tight_layout()
plt.show()

## View 2: make the handoff contract visible

GeoJSON point positions use WGS84 longitude then latitude in degrees. We intentionally export **two-dimensional geometry**: GeoJSON's optional third coordinate is altitude in meters, while USGS's source third value is depth in kilometers positive downward. Placing that depth directly into an altitude coordinate would invert and mis-scale it. Depth remains an explicitly named property. A receiving application can construct z = −1000 × depth_km meters with an explicit vertical-reference caveat; the catalog depth reference is not a surveyed terrain elevation.

The standalone [earthquake story](https://jltobias.github.io/JupyterLite-Data-Visualization/repository-examples/) draws an original canvas scene and keeps an HTML table/download path available. Canvas perspective does not require WebGL. No remote basemap, imagery or native GIS package is needed for its default view.

In [ ]:
handoff_metadata = {
    "title": "January 2024 Japan USGS teaching selection, magnitude >=5",
    "source": rx.manifest()["files"]["usgs/events.geojson"]["url"],
    "retrieved_utc": rx.manifest()["retrieved_utc"],
    "input_sha256": rx.manifest()["files"]["usgs/events.geojson"]["sha256"],
    "exported_count": len(selected),
    "selection": {"start_utc": "2024-01-01T00:00:00Z", "end_utc_exclusive": "2024-02-01T00:00:00Z",
                  "min_magnitude": 5.0, "catalog_min_magnitude": 4.5, "bbox_wgs84": [128, 30, 148, 46]},
    "terms": rx.manifest()["sources"]["usgs"]["terms"],
    "terms_url": rx.manifest()["sources"]["usgs"]["terms_url"],
    "credit": "U.S. Geological Survey Earthquake Hazards Program",
    "geometry": "WGS84 longitude, latitude degrees; two-dimensional coordinates",
    "depth": "depth_km property is catalog depth, km positive down; not GeoJSON altitude",
    "runtime": "Created using pandas/stdlib in CPython or Pyodide; external GIS import is a separate step",
    "limitations": "Bounded catalog; revisions and mixed magnitude types; no prediction, fault surface or depth uncertainty model",
}
features = []
for _, row in selected.iterrows():
    features.append({"type": "Feature", "id": row.id,
                     "geometry": {"type": "Point", "coordinates": [float(row.longitude), float(row.latitude)]},
                     "properties": {"time_utc": row.time_utc.isoformat(), "magnitude": float(row.magnitude),
                                    "magnitude_type": row.magnitude_type, "depth_km": float(row.depth_km),
                                    "place": row.place, "source_url": row.source_url}})
handoff = {"type": "FeatureCollection", "metadata": handoff_metadata, "features": features}
assert len(features) == len(selected) and all(len(f["geometry"]["coordinates"]) == 2 for f in features)
display(handoff_metadata)
display(HTML(selected[["id", "time_utc", "magnitude", "magnitude_type", "depth_km", "source_url"]].to_html(index=False, render_links=True, escape=True)))
fig, ax = plt.subplots(figsize=(11, 3))
steps = [f"78 packaged catalog records", f"M≥5: {len(selected)} events", "2D GeoJSON + named depth", "External GIS import"]
for i, label in enumerate(steps):
    ax.text(i, 0, label, ha="center", va="center", fontsize=9,
            bbox={"boxstyle": "round,pad=.7", "facecolor": "#e1edf0" if i < 3 else "#f4e4cc", "edgecolor": "#7796a3"})
    if i < 3:
        ax.annotate("", (i+.72, 0), (i+.28, 0), arrowprops={"arrowstyle": "->", "color": "#527684"})
ax.set(xlim=(-.6, 3.6), ylim=(-1, 1), title="Inspectable exchange chain: the external runtime starts at the last box")
ax.axis("off")
fig.tight_layout()
plt.show()

## Export: create new artifacts, never overwrite observations

The next cell writes small GeoJSON, CSV, HTML and provenance JSON artifacts in a separate `exports/` directory. Exclusive creation and numbered suffixes preserve earlier exports. Each download link contains the bytes directly, so downloading works even when the kernel's browser filesystem is not exposed as an HTTP file route. In JupyterLite, download the artifacts to persist them outside browser storage; the Book shows the same prepared links.

The HTML evidence summary is self-contained and uses no external scripts, services or imagery. The GIS geometry deliberately retains no scene exaggeration.

In [ ]:
csv_text = selected.to_csv(index=False)
geojson_text = json.dumps(handoff, ensure_ascii=False, indent=2)
# Verify the actual serialized artifact, including coordinate order and depth.
emitted = json.loads(geojson_text)
assert len(emitted["features"]) == emitted["metadata"]["exported_count"] == len(selected)
assert emitted["metadata"]["input_sha256"] == rx.manifest()["files"]["usgs/events.geojson"]["sha256"]
for feature, (_, source_row) in zip(emitted["features"], selected.iterrows()):
    assert feature["id"] == source_row.id
    assert feature["geometry"]["type"] == "Point"
    assert feature["geometry"]["coordinates"] == [float(source_row.longitude), float(source_row.latitude)]
    assert feature["properties"]["depth_km"] == float(source_row.depth_km)
summary_html = "<!doctype html><html lang='en'><meta charset='utf-8'><title>USGS evidence handoff</title><h1>USGS January 2024 Japan: magnitude ≥5</h1><p>Original evidence summary. Depth km positive down; WGS84 lon/lat degrees. Bounded event catalog, not a forecast. Native GIS import is external.</p><pre>" + escape(json.dumps(handoff_metadata, indent=2)) + "</pre>" + selected.to_html(index=False, escape=True) + "</html>"
artifacts = {"earthquake-handoff.geojson": (geojson_text, "application/geo+json"),
             "earthquake-handoff.csv": (csv_text, "text/csv"),
             "earthquake-evidence.html": (summary_html, "text/html"),
             "earthquake-provenance.json": (json.dumps(handoff_metadata, indent=2), "application/json")}
export_dir = lesson_dir / "exports"
export_dir.mkdir(exist_ok=True)
for name, (text, mime) in artifacts.items():
    target = export_dir / name
    suffix = 1
    while target.exists():
        target = export_dir / f"{Path(name).stem}-{suffix}{Path(name).suffix}"
        suffix += 1
    with target.open("x", encoding="utf-8", newline="") as stream:
        stream.write(text)
    encoded = base64.b64encode(text.encode("utf-8")).decode("ascii")
    display(HTML(f'<p><a download="{escape(target.name)}" href="data:{mime};base64,{encoded}">Download {escape(target.name)}</a> ({len(text.encode("utf-8")):,} bytes)</p>'))
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(list(artifacts), [len(value[0].encode("utf-8"))/1024 for value in artifacts.values()], color="#377b92")
rx.finish(ax, "Small handoff artifacts derived from the same selection", "Encoded text size (KiB, before data-link base64)", "Artifact")
fig.tight_layout()
plt.show()

## View 3: choose the receiving runtime deliberately

| Destination | What to do with this export | Boundary |
| --- | --- | --- |
| Current Pyodide notebook / canvas story | Inspect the table, geographic positions and derived scene | No native GIS, GPU or remote viewer required |
| JupyterGIS environment | Import the two-dimensional GeoJSON as a vector source and inspect layer/CRS settings | Separate JupyterGIS installation/runtime; no `.jGIS` compatibility claim is made here |
| Browser MapLibre or GeoLibre viewer | Load the small GeoJSON using that viewer's supported file interface | Separate application; hosted viewers and basemaps may need network and terms checks |
| Native RustGIS, QGIS or GDAL | Import GeoJSON and select a suitable projected CRS before distance work | Native CPython/application workflow outside this browser kernel |
| Native GeoAI / PyTorch | Establish an appropriate scientific task, data rights and spatial validation first | Separate native/GPU environment; this export is not a trained model |

An exchange artifact is concrete and reviewable without claiming every external application has been installed or tested. The atlas below preserves the exact source connections behind the handoff discussion.

In [ ]:
display(HTML(atlas.loc[atlas.lessons.astype(str).str.contains("09"),
                      ["repository", "connection", "rights_evidence", "readme_url"]].to_html(index=False, render_links=True, escape=True)))

## Exercise

Import a downloaded copy into an external GIS you already use. Check longitude/latitude order, count, units and the depth property against the evidence table. Record the application/version, any CRS conversion and whether altitude was constructed. If no external GIS is available, inspect the GeoJSON text and compare its first feature with the table. Rewrite one story panel to state both its claim and its principal limitation.

## Limitations and reflection

The exported events are a filtered selection of a bounded USGS catalog, with revisions and mixed magnitude types. Two-dimensional GeoJSON avoids an incorrect depth/altitude convention but cannot encode a geological surface. The canvas scene uses a documented local approximation; external precision work requires a suitable CRS and vertical reference. Native RustGIS, GeoAI, GDAL, QGIS and JupyterGIS integration are outside the current Pyodide environment. No ABM, health-risk model, calibrated digital twin or full GIS workspace integration is claimed.

## Citations and repository connections

[USGS data format](https://earthquake.usgs.gov/earthquakes/feed/v1.0/geojson.php) and [credit guidance](https://www.usgs.gov/information-policies-and-instructions/copyrights-and-credits); [GeoJSON position convention, RFC 7946 §3.1.1](https://www.rfc-editor.org/rfc/rfc7946#section-3.1.1). Pinned story/handoff references are listed in the atlas output above, including GeoLibre, RustGIS, Gaborone, DRC, Global Health GIS and Rewiring Global Health. The actual exported fields, story and visualizations are original.

See the [companion index](index.md) and [full attribution](https://jltobias.github.io/JupyterLite-Data-Visualization/book/references.html). These are original teaching code and narrative; linked repository notebooks and authored figures have not been copied.